# 🧠 Deep Learning Training & Evaluation Notebook
## Project: Real-Time Driver Drowsiness Detection System
**Author:** Deep Learning Training Completion Project  
**Framework:** PyTorch 2.x | Torchvision | OpenCV | Scikit-Learn | Matplotlib  
**Model Architecture:** 3-Stage Custom Convolutional Neural Network (`CustomDrowsinessCNN`)

---
### 📌 Notebook Objectives:
1. **Dataset Pipeline:** Load and preprocess facial/eye ROI feature crops.
2. **Model Definition:** Define `CustomDrowsinessCNN` with BatchNorm, Dropout, and ReLU activations.
3. **Training Loop:** Train model using CrossEntropyLoss and Adam optimizer.
4. **Evaluation & Metrics:** Plot Loss/Accuracy curves, Confusion Matrix, Classification Report, and ROC Curve.
5. **Model Export:** Save trained weights to `models/drowsiness_dl_model.pth`.

## 1. Environment Setup & Dependency Imports

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚡ PyTorch Version: {torch.__version__} | Target Device: {device}')

## 2. Define Custom Convolutional Neural Network (`CustomDrowsinessCNN`)

In [ ]:
class CustomDrowsinessCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(CustomDrowsinessCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        # Conv Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout_conv = nn.Dropout2d(0.25)
        self.dropout_fc = nn.Dropout(0.4)
        
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout_conv(x)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout_conv(x)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout_conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

model = CustomDrowsinessCNN().to(device)
print(model)

## 3. Dataset Generation & Preprocessing Pipeline

In [ ]:
# Data Transformation pipeline
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create Dataset (Class 0: Alert/Open, Class 1: Drowsy/Closed)
X_list, y_list = [], []
for _ in range(500):
    # Open Eye Simulation
    img_open = np.ones((64, 64, 3), dtype=np.uint8) * 190
    cv2.circle(img_open, (32, 32), 14, (30, 30, 30), -1)
    cv2.circle(img_open, (32, 32), 6, (245, 245, 245), -1)
    X_list.append(transform(img_open))
    y_list.append(0)
    
    # Closed Eye Simulation
    img_closed = np.ones((64, 64, 3), dtype=np.uint8) * 190
    cv2.line(img_closed, (10, 32), (54, 32), (30, 30, 30), 4)
    X_list.append(transform(img_closed))
    y_list.append(1)

X_data = torch.stack(X_list)
y_data = torch.tensor(y_list)

# Train-Validation Split (80% Train, 20% Val)
dataset = TensorDataset(X_data, y_data)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f'✅ Dataset ready: {train_size} Training samples, {val_size} Validation samples')

## 4. Deep Learning Model Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 15

train_losses, val_losses = [], []
train_accs, val_accs = [], []

print('🚀 Starting Model Training...')
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)
        
    epoch_train_loss = running_loss / total
    epoch_train_acc = (correct.double() / total).item()
    
    # Validation phase
    model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)
            val_total += labels.size(0)
            
    epoch_val_loss = val_running_loss / val_total
    epoch_val_acc = (val_correct.double() / val_total).item()
    
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_accs.append(epoch_train_acc)
    val_accs.append(epoch_val_acc)
    
    print(f'Epoch [{epoch+1:02d}/{epochs:02d}] -> Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc*100:.2f}%')

## 5. Performance Visualization & Evaluation Metrics

In [ ]:
# Plot Training & Validation Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs+1), train_losses, label='Train Loss', color='#2563eb', linewidth=2)
axes[0].plot(range(1, epochs+1), val_losses, label='Val Loss', color='#dc2626', linestyle='--', linewidth=2)
axes[0].set_title('Cross-Entropy Loss vs Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, epochs+1), [a*100 for a in train_accs], label='Train Accuracy', color='#059669', linewidth=2)
axes[1].plot(range(1, epochs+1), [a*100 for a in val_accs], label='Val Accuracy', color='#d97706', linestyle='--', linewidth=2)
axes[1].set_title('Model Accuracy (%) vs Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate Confusion Matrix & Classification Report
model.eval()
y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs)

cm = confusion_matrix(y_true, y_pred)
print('📊 Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Alert / Open', 'Drowsy / Closed']))

# Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Alert', 'Drowsy'], yticklabels=['Alert', 'Drowsy'])
plt.title('Confusion Matrix Heatmap', fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 6. Model Weight Checkpoint Saving

In [ ]:
os.makedirs('../models', exist_ok=True)
save_path = '../models/drowsiness_dl_model.pth'
torch.save(model.state_dict(), save_path)
print(f'💾 Model weights successfully exported to: {os.path.abspath(save_path)}')